# 시나리오2 — 새벽항로: 문서 미리보기 서버의 야간 사건

**HC-2026-0715 · 독립형 클라우드 침해사고 조사 · 실습 5시간**

여러분은 HarborCart의 사고 조사 담당자입니다. 한 대의 EC2 인스턴스에서 시작한 이상 징후를 조사해, 어떤 자격증명이 사용되었고 어떤 자료가 반환되었으며 대응이 효과가 있었는지 설명해야 합니다.

이 노트북은 공개 침해사고에서 관찰된 행위 유형을 참고해 만든 **새로운 합성 사건**입니다. 실제 피해자의 원본 로그가 아닙니다. IP·회사·계정·키 식별자는 모두 연습용이며 실제 자격증명이나 공격 실행 코드는 포함하지 않습니다. AWS 계정·유료 서비스·이전 수업 자료는 필요하지 않습니다.

도착점은 **근거가 붙은 사건 타임라인, 데이터 영향 집계, 헌팅 룰 4개, 대응 검증, 최종 보고서**입니다. 코드가 실행된 것과 사건의 결론이 입증된 것은 다릅니다.

## 0. 파일을 처음 여는 방법 · 20분

1. 학생 배포 ZIP을 PC에서 압축 해제합니다. `.ipynb`와 `새벽항로_evidence.zip`을 찾습니다. 증거 ZIP은 다시 풀 필요가 없습니다.
2. [Google Colab](https://colab.research.google.com/) → **파일 → 노트북 업로드**에서 이 `.ipynb`를 선택합니다. **파일 → Drive에 사본 저장**으로 개인 답안 사본을 만듭니다.
3. **연결**을 누릅니다. 왼쪽 **파일(폴더 아이콘)** 창에 `새벽항로_evidence.zip`을 업로드하고 완료될 때까지 기다립니다.
4. 코드 셀은 **▶ 또는 Shift+Enter**로 실행합니다. 설명 셀은 더블클릭해 답을 적습니다. 코드를 바꾼 뒤에는 해당 셀을 다시 실행합니다.
5. 아래 준비 셀부터 실행합니다. 처음부터 모두 실행하지 말고 장별로 진행합니다. 새 런타임에서는 증거 ZIP 업로드와 준비 셀 실행부터 다시 시작합니다.

로컬 Jupyter라면 노트북과 증거 ZIP을 같은 폴더에 놓습니다. Python 3.10 이상, pandas 2.2 이상, IPython이 필요합니다. 데이터는 ZIP 안에서 직접 읽으며 추출·수정하지 않습니다.

처음 연습: 아래 `수사관 이름`을 자신의 이름으로 바꾸고 다시 실행하세요.

검토판 진행 안내: 요청 ID나 키를 바꾸면 이전 조사 결과를 비웁니다. H1 정의→검증→요청 선택, H2 정의→검증→키 선택, H3 정의→검증, 추가 자료 공개, H4 정의→검증 순서로 실행하세요. 함수만 수정하고 검증 셀을 실행하지 않으면 완료로 표시하지 않습니다. 첫 셀부터 순서대로 실행하는 기본 경로에서도 TODO는 미완료 안내로 남습니다.

완료 목표는 제공 범위의 사건 연결·객체 반환량·대응 관찰입니다. 최초 취약 입력, 실제 피해 인원, 예약 작업의 실행 여부, 모든 영속화 제거는 추가 증거 필요로 분류하며 답을 얻을 때까지 멈추지 않습니다.

In [ ]:
INVESTIGATOR = '수사관 이름'
print(f'{INVESTIGATOR}님, 사건 공책이 열렸습니다.')

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json, hashlib, html
import pandas as pd
from IPython.display import display, HTML

roots = [Path.cwd(), Path.cwd()/'학생용', Path.cwd().parent/'학생용', Path('/content')]
archive = next((p/'새벽항로_evidence.zip' for p in roots if (p/'새벽항로_evidence.zip').is_file()), None)
if archive is None:
    raise FileNotFoundError('Colab 파일 창에 새벽항로_evidence.zip을 올리거나, 로컬에서 노트북과 같은 폴더에 놓으세요.')
evidence = ZipFile(archive)
def read_json(name):
    return json.loads(evidence.read(name))
def read_rows(name):
    return [json.loads(line) for line in evidence.read(name).decode('utf-8').splitlines() if line.strip()]

manifest = read_json('manifest.json')
bad = [item['path'] for item in manifest['files'] if hashlib.sha256(evidence.read(item['path'])).hexdigest() != item['sha256']]
assert not bad, f'증거 무결성 확인 실패: {bad}'
case = read_json('context/case_brief.json')
assets = pd.DataFrame(read_json('context/assets.json'))
catalog = pd.DataFrame(read_json('context/object_catalog.json'))
expected_sources = read_json('context/expected_sources.json')
changes = pd.DataFrame(read_json('context/change_calendar.json'))
coverage = read_json('context/collection_scope.json')
permission_review = read_json('context/permission_review.json')
print('증거 ZIP 확인 완료 / 사건:', case['case_id'], '/ 파일:', len(manifest['files']))
print('해시 일치는 배포 이후 동일성을 확인합니다. 내용의 현실성이나 진실성을 증명하지는 않습니다.')
progress = {name: '미완료' for name in ['H1', 'H2', 'H3', 'H4']}
analysis_state = {'request':None, 'key':None, 'initial_signature':None, 'final_signature':None, 'containment_signature':None, 'release':False}
initial_reads = final_reads = impact_comparison = containment = None
def current_signature():
    key = globals().get('FOCUS_KEY')
    ips = globals().get('external_ips', [])
    if key and ips and analysis_state['key']==key and progress['H2']=='통과':
        return (key,tuple(sorted(ips)))
    return None
def release_ready():
    return bool(globals().get('FOLLOWUP_RELEASED',False) and analysis_state['release'] and isinstance(globals().get('all_ct'),pd.DataFrame))

## 1. 신고 접수와 조사 범위 · 15분

HarborCart는 상품 문서를 이미지로 바꾸는 미리보기 서비스를 운영합니다. ALB가 요청을 받아 두 render 인스턴스로 전달합니다. 인스턴스의 외부 API 호출은 NAT 주소로 보이며, 웹 로그의 `client_ip`는 신뢰하는 ALB에서 전달받은 원래 클라이언트 주소입니다. 두 주소의 의미를 혼동하지 마세요.

새벽 01:32 KST, 미리보기 오류와 인스턴스 역할의 낯선 네트워크 사용 경보가 접수되었습니다. 현 시점에는 침해 규모나 공격 성공 여부를 모릅니다.

먼저 아래 자산과 신고 내용을 읽고 **침해 가설 1개, 정상 대안 1개, 이를 구분할 증거 1개**를 적습니다. 모든 로그의 원래 시각은 UTC이며 표의 `time_kst`는 표시용입니다.

침해 가설:

정상 대안:

먼저 확인할 증거와 이유:

In [ ]:
print(case['triage']['message'])
print('신고 시각 KST:', pd.Timestamp(case['triage']['time']).tz_convert('Asia/Seoul'))
FOCUS_INSTANCE = case['triage']['instance_id']
display(assets[['name','instance_id','role','private_ip','approved_egress']])
print('정상 비교 기준선 종료:', pd.Timestamp(case['baseline_end']))
print('초기 사건 자료는', coverage['initial_event_end'], '까지 제공됩니다.')

## 2. 데이터 품질: 같은 이벤트가 두 번 도착했다면 · 15분

CloudTrail의 중첩 JSON은 분석하기 쉽게 펼쳐 줍니다. `record_id`는 원래 `eventID`, `access_key_id`는 `userIdentity.accessKeyId`입니다. 식별자 이름을 바꾸어도 의미는 유지합니다. 전체 대응표는 데이터 사전에 있습니다.

수집기 재전송 때문에 같은 이벤트가 중복될 수 있습니다. **CloudTrail eventID**, **S3 request_id**를 기준으로 중복을 제거합니다. 동일 객체라도 새 request_id로 다시 읽었다면 별도의 전송입니다. 시간 순서는 수집 시각이 아닌 이벤트 시각으로 확인합니다.

In [ ]:
def normalize_cloud(rows):
    normalized = []
    for r in rows:
        identity = r.get('userIdentity', {})
        session = identity.get('sessionContext', {})
        issuer = session.get('sessionIssuer', {})
        params = r.get('requestParameters') or {}
        normalized.append({
            'record_id': r['eventID'], 'request_id': r['requestID'], 'event_time': r['eventTime'],
            'collected_at': r['_collected_at'], 'event_name': r['eventName'], 'event_source': r['eventSource'],
            'error_code': r.get('errorCode', ''), 'source_ip': r['sourceIPAddress'],
            'identity_type': identity.get('type',''), 'principal': identity.get('arn',''),
            'role': issuer.get('userName',''), 'session_name': identity.get('arn','').rsplit('/',1)[-1],
            'access_key_id': identity.get('accessKeyId',''), 'session_created_at': session.get('attributes',{}).get('creationDate',''),
            'bucket': params.get('bucketName',''), 'object_key': params.get('key',''),
            'category': r['eventCategory'], 'request_parameters': params,
        })
    result = pd.DataFrame(normalized)
    result['event_time'] = pd.to_datetime(result.event_time, utc=True)
    result['time_kst'] = result.event_time.dt.tz_convert('Asia/Seoul')
    return result

def load_phase(phase):
    frames = {}
    for name in ['cloudtrail','web','host','auth','vpc_flow','s3_access','logging_status']:
        member = f'{phase}/{name}.jsonl'
        if member not in evidence.namelist():
            frames[name] = pd.DataFrame()
            continue
        rows = read_rows(member)
        if name == 'cloudtrail':
            table = normalize_cloud(rows)
        else:
            table = pd.DataFrame(rows)
            table['event_time'] = pd.to_datetime(table.time, utc=True)
            table['time_kst'] = table.event_time.dt.tz_convert('Asia/Seoul')
        frames[name] = table
    return frames

initial = load_phase('initial')
ct_raw, s3_raw = initial['cloudtrail'], initial['s3_access']
ct = ct_raw.drop_duplicates('record_id').copy()
s3 = s3_raw.drop_duplicates('request_id').copy()
web, host, flow, auth = [initial[name] for name in ['web','host','vpc_flow','auth']]
display(pd.DataFrame([{'로그':name,'수신 행':len(table)} for name,table in initial.items()]))
print('CloudTrail 중복:', len(ct_raw)-len(ct), '/ S3 중복:', len(s3_raw)-len(s3))
print('첫 이벤트:', ct.event_time.min(), '/ 마지막 이벤트:', ct.event_time.max())
print('호출 결과는 오류 코드와 해당 서비스 응답을 함께 확인합니다.')

확인한 중복 수와 제거 키:

새로운 request_id로 같은 객체를 두 번 읽으면 전송량을 어떻게 계산할까요?

노트북의 UTC와 KST가 다른 날짜로 표시되는 이유:

## 3. 3분 탐색: 가장 시끄러운 IP가 범인일까 · 20분

다음 셀은 HTTP 오류가 많은 주소를 바로 보여 줍니다. **먼저 조사할 주소를 고르고 이유를 쓴 뒤** 다음 변경 일정을 공개하세요. 오류가 많다는 이유만으로 악성이라고 결론내리지 않습니다.

In [ ]:
day_web = web[web.event_time.ge(pd.Timestamp(case['analysis_day']))].copy()
web_summary = day_web.assign(server_error=day_web.status.ge(500)).groupby('client_ip').agg(
    requests=('record_id','size'), server_errors=('server_error','sum')).reset_index().sort_values('server_errors',ascending=False)
display(web_summary.head(8))
def show_bars(table, label, value):
    peak = max(float(table[value].max()), 1)
    bars = ''.join(f'<div style="margin:6px 0"><span style="display:inline-block;width:145px">{html.escape(str(r[label]))}</span><span style="display:inline-block;background:#24798a;width:{240*float(r[value])/peak:.0f}px;height:14px"></span> {int(r[value])}</div>' for _, r in table.iterrows())
    display(HTML(bars))
show_bars(web_summary.head(6),'client_ip','server_errors')

공개 전 예상 — 먼저 조사할 주소와 이유:

이제 승인된 변경 내역과 네트워크 차단 기록을 봅니다.

In [ ]:
display(changes)
display(flow.groupby(['action','dstport']).size().rename('flow_records').reset_index())
print('REJECT는 네트워크 차단 관찰입니다. 이 기록만으로 SSH 로그인 성공을 주장하지 않습니다.')

정상 대안으로 설명할 수 있는 활동:

여전히 남는 의문:

네트워크 스캔과 인스턴스 실행 흔적을 구분하는 데 필요한 로그:

## 4. H1: 웹 요청이 프로세스로 이어졌는가 · 40분

첫 번째 룰을 직접 작성합니다. `find_request_shells(process_events)`는 다음을 **모두** 만족하는 원래 행을 DataFrame으로 반환해야 합니다.

- `event == 'process_start'`
- 부모 프로세스가 `render-worker`
- 실행 프로세스가 `sh` 또는 `bash`
- 실행 사용자가 `render`
- `request_id`가 빈 값이 아님: 외부 요청 맥락이 있는 경우

이 룰은 조사 후보를 찾습니다. 정상 서비스가 의도적으로 셸을 사용하는지 업무 확인이 필요합니다. 단순한 모든 셸 탐지는 승인된 점검 작업도 잡습니다.

함수 안의 `raise NotImplementedError(...)`를 지우고 구현하세요. **함수 정의 셀 → 검증 셀** 순서로 다시 실행합니다. 검증 셀은 바꾸지 않습니다.

<details><summary>힌트</summary>각 비교를 괄호로 묶어 `&`로 연결합니다. 여러 후보 값은 `.isin(...)`, 결측·빈 요청 ID는 `.fillna('').ne('')`로 다룹니다.</details>

In [ ]:
progress['H1']='미완료'
analysis_state['request']=None
def find_request_shells(process_events):
    # TODO: 다섯 조건을 모두 만족하는 행을 반환하세요.
    raise NotImplementedError('H1: 요청 맥락이 있는 셸 실행 룰을 작성하세요.')

In [ ]:
progress['H1'] = '미완료'
shell_roots = pd.DataFrame()
analysis_state['request'] = None
h1_test = pd.DataFrame([
    ['p1','process_start','render-worker','sh','render','req-x'],
    ['p2','process_start','render-worker','sh','render',''],
    ['p3','process_start','systemd','sh','root','req-x'],
    ['p4','process_start','render-worker','convert','render','req-x'],
    ['p5','metadata_access','render-worker','sh','render','req-x'],
    ['p6','process_start','render-worker','bash','render','req-y'],
    ['p7','process_start','render-worker','sh','render',None],
    ['p8','process_start','render-worker','sh','other','req-x'],
],columns=['record_id','event','parent_process','process','user','request_id'])
try:
    h1_answer = find_request_shells(h1_test.copy())
    assert isinstance(h1_answer,pd.DataFrame) and h1_answer.record_id.tolist()==['p1','p6'], '프로세스 종류, 부모, 사용자, 요청 맥락을 함께 확인하세요.'
    shell_roots = find_request_shells(host.copy())
    display(shell_roots[['record_id','time_kst','instance_id','process','parent_process','pid','ppid','request_id']])
    progress['H1'] = '통과'
except NotImplementedError as pending:
    print('미완료 —',pending)

표에서 조사할 `request_id`를 복사해 아래 `None`을 문자열로 바꾸세요. 예: `FOCUS_REQUEST = '표에서 복사한 값'`. UUID 자체를 추측할 필요는 없습니다.

In [ ]:
FOCUS_REQUEST = None  # H1 결과의 request_id를 복사해 입력
FOCUS_REQUEST = FOCUS_REQUEST.strip() if isinstance(FOCUS_REQUEST,str) else None
request_web = request_host = None
analysis_state['request'] = None
if FOCUS_REQUEST and progress['H1']=='통과' and FOCUS_REQUEST in set(shell_roots.request_id):
    request_web = web[web.request_id.eq(FOCUS_REQUEST)].copy()
    request_host = host[host.request_id.eq(FOCUS_REQUEST)].sort_values('event_time').copy()
    display(request_web[['record_id','time_kst','client_ip','path','status','request_id']])
    display(request_host[['record_id','time_kst','event','process','parent_process','pid','ppid','detail']])
    if not request_web.empty and not request_host.empty:
        analysis_state['request'] = FOCUS_REQUEST
    else:
        print('두 로그에서 연결되지 않았습니다. 요청 ID의 공백·오타 또는 실제 연결 근거가 있는지 확인하세요.')
else:
    print('H1 정의·검증 후 표의 request_id를 입력하세요. 이전 요청 결과는 비웠습니다.')

분석 기록:

1. 웹 로그와 호스트 로그를 연결한 키 및 양쪽 record_id:
2. 가장 먼저 확인되는 비정상 실행 시각과 부모→자식 관계:
3. 메타데이터 접근을 정상 cloud-agent와 구분한 근거:
4. 사용자 범위의 작업 등록은 무엇을 의미하며, 작업이 실제 실행되었는지도 알 수 있나요?
5. 요청 본문·취약 코드가 없는데 특정 취약점이나 RCE 입력값을 확정할 수 있나요?

**직접 코드 추가:** 해당 인스턴스에서 같은 시간대의 `network_connect`와 VPC Flow Logs의 목적지·포트를 비교하세요. 웹 client_ip, NAT 주소, 외부 연결 목적지를 각각 적습니다. Flow Logs의 bytes는 페이로드 내용이나 자격증명 유출량을 증명하지 않습니다.

## 5. H2: 역할 이름 대신 세션과 키를 추적하기 · 40분

render-01과 render-02는 같은 역할을 사용합니다. 역할 이름만 묶으면 어느 인스턴스의 세션이 사용됐는지 잃습니다. `session_name`은 이 자료에서 인스턴스 ID이고, `access_key_id`는 임시 자격증명의 식별자입니다. 비밀 키는 제공하지 않습니다.

`hunt_external_sessions(events, expected_sources)`를 완성하세요. 다음 조건을 모두 만족하는 이벤트 행을 반환합니다.

- `identity_type == 'AssumedRole'`, `session_name`이 `i-`로 시작
- `role`이 승인된 출발지 목록의 키로 존재
- `source_ip`가 **해당 역할의** 승인 주소 목록에 없음

새 IP라는 이유만으로 범인이라고 하지 않습니다. 승인 내역·이전 출발지·호스트 실행 맥락을 함께 봅니다. 아직 모르는 역할은 이 룰의 대상에서 제외하고 별도 검토 대상으로 남깁니다.

<details><summary>힌트</summary>행의 role로 사전의 주소 목록을 조회한 뒤 source_ip가 들어 있는지 확인합니다. 모든 역할의 허용 IP를 하나로 합치면 다른 역할의 예외까지 적용됩니다. 원래 인덱스를 유지하는 bool Series가 유용합니다.</details>

In [ ]:
progress['H2']='미완료'
analysis_state['key']=None
initial_reads=final_reads=impact_comparison=containment=None
analysis_state['initial_signature']=analysis_state['final_signature']=analysis_state['containment_signature']=None
def hunt_external_sessions(events, expected_sources):
    # TODO: 알려진 EC2 역할 세션 중 해당 역할의 승인 IP 밖에서 사용된 행을 반환하세요.
    raise NotImplementedError('H2: 역할별 출발지 조건을 구현하세요.')

In [ ]:
progress['H2'] = '미완료'
external = pd.DataFrame()
analysis_state['key'] = None
analysis_state['initial_signature'] = analysis_state['final_signature'] = analysis_state['containment_signature'] = None
initial_reads = final_reads = impact_comparison = containment = None
h2_test = pd.DataFrame([
    ['e1','AssumedRole','i-one','Render','192.0.2.10'],
    ['e2','AssumedRole','i-one','Render','192.0.2.20'],
    ['e3','AssumedRole','i-two','Backup','192.0.2.20'],
    ['e4','AssumedRole','analyst-session','Render','192.0.2.99'],
    ['e5','IAMUser','i-one','Render','192.0.2.99'],
    ['e6','AssumedRole','i-x','Unknown','192.0.2.99'],
],columns=['record_id','identity_type','session_name','role','source_ip'],index=[2,4,6,8,10,12])
try:
    h2_answer = hunt_external_sessions(h2_test.copy(), {'Render':['192.0.2.10'],'Backup':['192.0.2.20']})
    assert isinstance(h2_answer,pd.DataFrame) and h2_answer.record_id.tolist()==['e2'], '역할별 승인 IP와 EC2 세션 조건을 확인하세요.'
    external = hunt_external_sessions(ct.copy(), expected_sources)
    display(external.groupby(['role','session_name','access_key_id','source_ip']).agg(
        calls=('record_id','size'),first=('time_kst','min'),last=('time_kst','max')).reset_index())
    progress['H2'] = '통과'
except NotImplementedError as pending:
    print('미완료 —',pending)

표에서 조사할 access_key_id를 아래에 입력합니다. 외부 주소는 H2 룰 결과에서 가져옵니다. 정답 IP를 사전에 코드에 넣어 탐지한 것처럼 만들지 마세요.

In [ ]:
FOCUS_KEY = None  # H2 결과의 access_key_id를 복사해 입력
FOCUS_KEY = FOCUS_KEY.strip() if isinstance(FOCUS_KEY,str) else None
external_ips = []
key_timeline = None
analysis_state['key'] = None
analysis_state['initial_signature'] = analysis_state['final_signature'] = analysis_state['containment_signature'] = None
initial_reads = final_reads = impact_comparison = containment = None
if FOCUS_KEY and progress['H2']=='통과' and not external.empty and FOCUS_KEY in set(external.access_key_id):
    external_ips = external[external.access_key_id.eq(FOCUS_KEY)].source_ip.unique().tolist()
    key_timeline = ct[ct.access_key_id.eq(FOCUS_KEY)].sort_values('event_time')
    analysis_state['key'] = FOCUS_KEY
    display(key_timeline[['record_id','time_kst','session_name','source_ip','event_name','error_code','bucket','object_key']])
    print('조사할 외부 주소:',external_ips)
    print('현재 키로 H3 검증 셀을 다시 실행하세요. 이전 피해 집계는 비웠습니다.')
else:
    print('현재 H2 결과에서 이 키를 찾을 수 없습니다. H2 정의·검증 후 키를 확인하세요. 이전 결과는 비웠습니다.')

분석 기록:

- 동일 키가 정상 NAT와 승인되지 않은 주소에서 사용된 근거:
- 같은 역할을 쓰는 다른 인스턴스까지 침해됐다고 단정할 수 없는 이유:
- 이 세션을 특정 인간 사용자·공격자 실명과 동일시할 수 없는 이유:

**직접 코드 추가:** 선택한 키에서 관찰된 API를 이름·오류 코드별로 집계하세요. `CreateAccessKey`와 `StopLogging`은 각각 시도인가요, 확인된 성공인가요? 성공한 자료 접근과 실패한 제어 작업을 구분해 적습니다. EC2 세션의 MFA 표시를 사람의 콘솔 MFA 사용 여부로 해석하지 않습니다.

## 6. H3: 어떤 자료가 실제로 반환되었나 · 45분

CloudTrail은 호출 주체와 대상을, S3 접근 기록은 HTTP 상태와 전송 바이트를 제공합니다. `join_successful_reads(events, server_logs, key_id, source_ips)`를 구현하세요.

1. CloudTrail에서 선택한 키·조사 IP의 `GetObject`이며 `error_code == ''`인 호출을 고릅니다. `record_id` 중복을 제거합니다.
2. S3에서 `operation == 'REST.GET.OBJECT'`이고 `http_status == 200`인 전체 객체 응답을 고릅니다. `request_id` 중복을 제거합니다.
3. request_id로 연결한 뒤 **bucket, 객체 key, 요청 출발지**도 같은지 확인합니다.
4. `request_id, event_time, bucket, key, bytes_sent` 열을 가진 표를 반환합니다. 실제 재시도는 새로운 request_id이므로 남깁니다.

이 문제는 HTTP 200 전체 응답을 대상으로 합니다. 부분 응답(206)이 있는 사건은 범위별 바이트를 별도 분석해야 합니다. 이 사건의 성공한 객체 응답에는 206이 없습니다.

<details><summary>힌트 1</summary>`.drop_duplicates(...)`, `.merge(on='request_id', validate='one_to_one')`를 사용하세요. 합치기 전에 필요한 열만 고르고 서버 쪽 열 이름을 바꾸면 충돌을 줄일 수 있습니다.</details>
<details><summary>힌트 2</summary>CloudTrail 객체 열 이름은 object_key이고 S3는 key입니다. CloudTrail source_ip와 S3 remote_ip도 비교합니다. 반환할 때 object_key를 key로 바꿀 수 있습니다.</details>

In [ ]:
progress['H3']='미완료'
initial_reads=final_reads=impact_comparison=None
analysis_state['initial_signature']=analysis_state['final_signature']=None
def join_successful_reads(events, server_logs, key_id, source_ips):
    # TODO: 주체·결과 필터, 중복 제거, request_id 연결, 대상 교차 확인을 구현하세요.
    raise NotImplementedError('H3: 두 로그를 연결해 확인된 전체 객체 응답을 반환하세요.')

In [ ]:
progress['H3'] = '미완료'
initial_reads = final_reads = impact_comparison = None
analysis_state['initial_signature'] = analysis_state['final_signature'] = None
h3_ct = pd.DataFrame([
    ['c1','q1','GetObject','','K1','192.0.2.9','b','a'],
    ['c1','q1','GetObject','','K1','192.0.2.9','b','a'],
    ['c2','q2','GetObject','','K1','192.0.2.9','b','a'],
    ['c3','q3','GetObject','AccessDenied','K1','192.0.2.9','b','z'],
    ['c4','q4','GetObject','','K2','192.0.2.9','b','a'],
    ['c5','q5','GetObject','','K1','192.0.2.9','b','a'],
    ['c6','q6','GetObject','','K1','192.0.2.10','b','a'],
    ['c7','q7','GetObject','','K1','192.0.2.9','b','a'],
],columns=['record_id','request_id','event_name','error_code','access_key_id','source_ip','bucket','object_key'])
h3_ct['event_time'] = pd.Timestamp('2026-01-01T00:00:00Z')
h3_s3 = pd.DataFrame([
    ['q1','REST.GET.OBJECT',200,'b','a','192.0.2.9',100],
    ['q1','REST.GET.OBJECT',200,'b','a','192.0.2.9',100],
    ['q2','REST.GET.OBJECT',200,'b','a','192.0.2.9',100],
    ['q3','REST.GET.OBJECT',403,'b','z','192.0.2.9',243],
    ['q4','REST.GET.OBJECT',200,'b','a','192.0.2.9',100],
    ['q5','REST.GET.OBJECT',200,'b','wrong-object','192.0.2.9',100],
    ['q6','REST.GET.OBJECT',200,'b','a','192.0.2.10',100],
    ['q7','REST.GET.OBJECT',200,'b','a','192.0.2.88',100],
],columns=['request_id','operation','http_status','bucket','key','remote_ip','bytes_sent'])
try:
    h3_answer = join_successful_reads(h3_ct.copy(),h3_s3.copy(),'K1',['192.0.2.9'])
    assert set(['request_id','event_time','bucket','key','bytes_sent']).issubset(h3_answer.columns), '지정된 반환 열을 확인하세요.'
    assert len(h3_answer)==2 and set(h3_answer.request_id)=={'q1','q2'} and h3_answer.bytes_sent.sum()==200, '수집 중복, 실제 재시도, 다른 주체, HTTP 결과, 대상·출발지 일치를 확인하세요.'
    progress['H3'] = '통과'
    if current_signature() is not None:
        initial_reads = join_successful_reads(ct.copy(),s3.copy(),FOCUS_KEY,external_ips)
        analysis_state['initial_signature'] = current_signature()
        display(initial_reads.assign(time_kst=initial_reads.event_time.dt.tz_convert('Asia/Seoul')))
        print('응답 수:',len(initial_reads),'/ 고유 객체:',len(initial_reads[['bucket','key']].drop_duplicates()))
        print('반환 바이트:',int(initial_reads.bytes_sent.sum()),'/ MiB:',round(initial_reads.bytes_sent.sum()/1024**2,4))
        display(initial_reads[['bucket','key']].drop_duplicates().merge(catalog,on=['bucket','key'],how='left'))
    else:
        print('H3 예시 검증 통과. 실제 조사를 위해 H2와 FOCUS_KEY를 완료하세요.')
except NotImplementedError as pending:
    print('미완료 —',pending)

분석 기록:

- 초기 자료에서 확인한 응답 수 / 고유 객체 수 / 반환 바이트:
- 재시도 때문에 바이트와 고유 객체 크기가 달라지는 이유:
- 반환된 정보의 분류와 업무 담당자:
- 접근이 차단된 객체를 피해 객체 목록에 더하면 안 되는 이유:
- S3의 반환 바이트로 실제 피해 고객 수나 공격자의 재배포 여부까지 알 수 있나요?

**직접 코드 추가:** 고유 객체의 카탈로그 크기 합과 모든 성공 응답의 bytes_sent 합을 나란히 출력하세요. 서로 다른 로그에 기록된 같은 전송을 중복 합산하지 않습니다. 외부에서 AWS API로 직접 받은 객체는 피해 인스턴스의 VPC Flow Logs에 나타나지 않을 수 있습니다.

## 7. 실패한 시도와 관찰 한계 · 20분

아래 기록으로 최초 가설을 수정하세요. 요청한 행위의 이름과 실제 결과를 구분해야 합니다. 비어 있는 responseElements를 보고 실패라고 단정하지도 않습니다.

In [ ]:
if current_signature() is not None:
    controls = ct[ct.access_key_id.eq(FOCUS_KEY) & ct.event_name.isin(['CreateAccessKey','StopLogging'])]
    display(controls[['record_id','time_kst','source_ip','event_name','error_code','request_parameters']])
    display(initial['logging_status'])
    display(pd.DataFrame(permission_review.items(),columns=['권한 검토 항목','확인 내용']))
else:
    print('먼저 조사할 키를 선택하세요.')
print('자료의 한계:')
for item in coverage['limitations']:
    print('-',item)

판단을 분리해 작성합니다.

| 주장 | 확인 / 반박 / 추가 증거 필요 | 근거 ID와 이유 |
|---|---|---|
| 새 IAM 키가 발급되었다 | | |
| 감사 로그가 중단되었다 | | |
| 사용자 범위의 예약 작업이 등록되었다 | | |
| 서버의 root 권한까지 획득했다 | | |
| 다른 render 인스턴스도 침해됐다 | | |

권한 검토에서 업무에 필요한 prefix와 실제 허용된 prefix는 어떻게 다른가요? 정상 NAT 승인 목록이 존재한다는 사실과 IAM에 출발지 네트워크 조건이 적용되어 있다는 사실을 구분하세요. 이 검토는 제공한 역할·버킷·API 범위에 한정됩니다.

성공하지 못한 공격 시도도 의도와 탐지 개선을 설명하는 근거가 됩니다.

## 8. 추가 자료 공개: 격리했으니 끝났을까 · 35분

**초기 영향과 대응 제안을 기록한 뒤 강사의 안내에 따라 진행합니다.** 아래 `FOLLOWUP_RELEASED`를 `True`로 바꿉니다. 초기 자료는 그대로 유지하고 이후 이벤트와 대응 기록을 더 읽습니다.

공개 전 예상 — 인스턴스의 네트워크를 격리하면 이미 외부로 복사된 임시 자격증명도 즉시 사용할 수 없게 될까요? 이유와 확인할 로그를 적습니다.

In [ ]:
FOLLOWUP_RELEASED = False
analysis_state['release'] = bool(FOLLOWUP_RELEASED)
analysis_state['final_signature'] = analysis_state['containment_signature'] = None
all_ct = all_s3 = actions = followup = final_reads = impact_comparison = containment = None
if FOLLOWUP_RELEASED:
    followup = load_phase('followup')
    all_ct = pd.concat([ct,followup['cloudtrail']],ignore_index=True).drop_duplicates('record_id')
    all_s3 = pd.concat([s3,followup['s3_access']],ignore_index=True).drop_duplicates('request_id')
    actions = pd.DataFrame(read_json('followup/response_actions.json'))
    actions['event_time'] = pd.to_datetime(actions.time,utc=True)
    actions['time_kst'] = actions.event_time.dt.tz_convert('Asia/Seoul')
    display(actions[['time_kst','action','target','status','note']])
    display(pd.DataFrame(read_json('followup/recovery_validation.json')['checks'].items(),columns=['확인 항목','결과']))
else:
    print('추가 자료 미공개. 초기 분석을 기록한 뒤 True로 바꾸세요.')

### H4. 대응 시점 이후의 동일 자격증명 사용을 확인하기

`check_containment(events, key_id, source_ips, isolated_at, revoked_at)`를 작성하세요.

선택한 키·조사 IP의 GetObject만 골라 record_id 중복을 제거합니다. 시간은 **대응 시각 이상(>=)**으로 비교합니다. 이 자료에서 GetObject의 빈 error_code는 오류 없는 호출, AccessDenied는 거부 호출이며, 성공 응답은 H3으로 교차 검증합니다.

반환 딕셔너리:

- `after_isolation_success`: 인스턴스 격리 이후 오류 없는 호출 수
- `after_revocation_success`: 세션 무효화 이후 오류 없는 호출 수
- `after_revocation_denied`: 세션 무효화 이후 AccessDenied 호출 수

<details><summary>힌트</summary>먼저 대상 호출을 고른 뒤 시각 조건과 오류 조건을 `&`로 연결합니다. bool Series의 `.sum()`으로 건수를 셀 수 있습니다. 다른 키와 다른 API를 섞지 마세요.</details>

In [ ]:
progress['H4']='미완료'
containment=None
analysis_state['containment_signature']=None
def check_containment(events, key_id, source_ips, isolated_at, revoked_at):
    # TODO: 대상 GetObject를 중복 제거하고 시점별 성공·거부 건수를 반환하세요.
    raise NotImplementedError('H4: 격리와 자격증명 대응을 구분해 검증하세요.')

In [ ]:
progress['H4'] = '미완료'
containment = None
analysis_state['containment_signature'] = None
h4_test = pd.DataFrame([
    ['a','09:59','GetObject','K','','192.0.2.9'],
    ['b','10:00','GetObject','K','','192.0.2.9'],
    ['c','10:10','GetObject','K','AccessDenied','192.0.2.9'],
    ['c','10:10','GetObject','K','AccessDenied','192.0.2.9'],
    ['d','10:11','GetObject','OTHER','','192.0.2.9'],
    ['e','10:11','ListBuckets','K','','192.0.2.9'],
    ['f','10:11','GetObject','K','','192.0.2.10'],
    ['g','10:10','GetObject','K','','192.0.2.9'],
],columns=['record_id','clock','event_name','access_key_id','error_code','source_ip'])
h4_test['event_time'] = pd.to_datetime('2026-01-01T'+h4_test.clock+':00Z',utc=True)
try:
    h4_answer = check_containment(h4_test.copy(),'K',['192.0.2.9'],pd.Timestamp('2026-01-01T10:00:00Z'),pd.Timestamp('2026-01-01T10:10:00Z'))
    assert h4_answer=={'after_isolation_success':2,'after_revocation_success':1,'after_revocation_denied':1}, '경계 시각, 중복, 다른 키·IP·API를 확인하세요.'
    progress['H4']='통과'
    if release_ready() and current_signature() is not None:
        isolated_at = actions.loc[actions.action.eq('isolate_instance'),'event_time'].iloc[0]
        revoked_at = actions.loc[actions.action.eq('revoke_prior_role_sessions'),'event_time'].iloc[0]
        containment = check_containment(all_ct.copy(),FOCUS_KEY,external_ips,isolated_at,revoked_at)
        analysis_state['containment_signature'] = current_signature()
        display(pd.DataFrame(containment.items(),columns=['대응 지표','건수']))
        display(all_ct[all_ct.access_key_id.eq(FOCUS_KEY) & all_ct.source_ip.isin(external_ips) & all_ct.event_time.ge(isolated_at)][['record_id','time_kst','event_name','error_code','object_key']])
    else:
        print('H4 예시 검증 통과. 추가 자료 공개와 키 선택을 마치세요.')
except NotImplementedError as pending:
    print('미완료 —',pending)

In [ ]:
if release_ready() and current_signature() is not None and progress['H3']=='통과' and analysis_state['initial_signature']==current_signature() and isinstance(initial_reads,pd.DataFrame):
    final_reads = join_successful_reads(all_ct.copy(),all_s3.copy(),FOCUS_KEY,external_ips)
    analysis_state['final_signature'] = current_signature()
    impact_comparison = pd.DataFrame([
        {'자료':'초기','성공 응답':len(initial_reads),'고유 객체':len(initial_reads[['bucket','key']].drop_duplicates()),'반환 바이트':int(initial_reads.bytes_sent.sum())},
        {'자료':'추가 자료 포함','성공 응답':len(final_reads),'고유 객체':len(final_reads[['bucket','key']].drop_duplicates()),'반환 바이트':int(final_reads.bytes_sent.sum())},
    ])
    display(impact_comparison)
    display(followup['logging_status'])
else:
    final_reads = impact_comparison = None
    analysis_state['final_signature'] = None
    print('현재 키로 H3 검증 셀을 실행한 뒤 추가 자료를 공개하세요. 이전 결과는 표시하지 않습니다.')

최종 판단:

1. 초기 피해 집계에서 무엇이 추가되었으며 어느 request_id가 근거인가요?
2. 호스트 격리와 세션 무효화가 해결하는 대상은 어떻게 다른가요?
3. 세션 무효화 후 성공 0건이라는 결과를 해석할 관찰 종료 시각·로그 범위는 무엇인가요?
4. 깨끗한 교체 인스턴스의 검증 기록과 기존 예약 작업에 대한 대응을 어떻게 보고할까요?
5. 재발 방지를 위한 앱 보완, 역할의 S3 권한 범위, 세션의 허용 네트워크, 로깅 유지 중 우선순위 3개와 이유를 적습니다.

단일 사건에서 룰이 작동한 사실은 운영 환경에서의 오탐률·탐지율을 측정한 결과가 아닙니다.

## 9. 사건을 끝까지 설명하기 · 50분

다음 순서로 **1쪽 보고서 + 근거 타임라인**을 완성합니다. 별도 `04_조사보고서_템플릿.md`를 사용해도 됩니다.

| 단계 | KST 시각 | 관찰 사실 | 근거 로그와 ID | 해석 / 확신 수준 |
|---|---|---|---|---|
| 최초 실행 | | | | |
| 역할 자격증명 접근 맥락 | | | | |
| 외부에서의 세션 사용 | | | | |
| 실제 객체 응답 | | | | |
| 실패한 제어 작업 | | | | |
| 인스턴스 격리 후 | | | | |
| 자격증명 대응 및 관찰 종료 | | | | |

### 사건 요약

무엇이 일어났나(3문장):

확인된 영향(인스턴스·세션·객체 수·응답 수·바이트):

실패하거나 확인되지 않은 행위:

초기 가설에서 바뀐 판단:

대응 결과와 관찰 범위:

남은 불확실성과 추가 수집할 증거:

재발 방지 우선순위 3개:

### 동료 검토

짝은 위 보고서에서 **근거를 넘은 주장 1개**, **빠진 근거 1개**, **잘 구분한 사실과 추정 1개**를 찾아 기록합니다. 피드백 후 보고서를 수정하세요. 실행 속도 대신 설명의 정확성과 판단 수정에 점수를 줍니다.

### 제출

노트북의 함수 코드·검증 출력·추가 코드·해석·타임라인을 개인 사본에 저장하고 `.ipynb`로 다운로드합니다. 파일 이름은 `HC_2026_0715_이름.ipynb`로 정합니다. 참고 정답이 아닌 자신의 증거 연결을 제출합니다.

In [ ]:
display(pd.DataFrame(progress.items(),columns=['룰','예시 코드 검증']))
sig = current_signature()
analysis_complete = bool(all(value=='통과' for value in progress.values()) and analysis_state['request'] and analysis_state['request']==FOCUS_REQUEST and sig is not None and release_ready() and analysis_state['initial_signature']==sig and analysis_state['final_signature']==sig and analysis_state['containment_signature']==sig)
print('필수 코드와 현재 선택의 실제 분석:', '완료 — 보고서·추가 코드·동료 검토는 별도 확인' if analysis_complete else '미완료 — 요청/키 선택 및 해당 검증 셀을 순서대로 실행하세요.')
print('근거 ID, 초기·최종 집계, 정상 대안, 미확인 사항, 관찰 범위와 동료 검토를 제출하세요.')

## 자료의 근거

공개 SCARLETEEL 조사에서 워크로드 침해 후 클라우드 권한이 악용되고 데이터가 탈취된 행위 유형을 참고했습니다. 이 실습의 회사·로그·수량·결과·대응 시점은 새로 설계했으며 실제 사건의 재현 기록이 아닙니다. [Sysdig 조사, 2023-02-28](https://www.sysdig.com/blog/cloud-breach-terraform-data-theft)

필드 의미 확인: [CloudTrail 이벤트](https://docs.aws.amazon.com/awscloudtrail/latest/userguide/cloudtrail-event-reference-record-contents.html), [CloudTrail 주체](https://docs.aws.amazon.com/awscloudtrail/latest/userguide/cloudtrail-event-reference-user-identity.html), [S3 접근 기록](https://docs.aws.amazon.com/AmazonS3/latest/userguide/LogFormat.html), [VPC Flow Logs](https://docs.aws.amazon.com/vpc/latest/userguide/flow-log-records.html). 자세한 합성 범위와 한계는 데이터 사전을 읽으세요.